# Week 5: Dynamic Mapping & Time-Machine Simulation

**Student Worksheet** — Fill in the code cells using AI assistance or your own code.

This week you will:
1. Call the CWA real-time rainfall API
2. Parse nested JSON into GeoDataFrame
3. Build interactive Folium maps with conditional styling
4. "Replay" Typhoon Fung-wong (2025) as a stress test
5. Overlay dynamic rainfall with shelter risk data

**Packages needed:** `geopandas`, `folium`, `requests`, `python-dotenv`, `branca`

## Cell [1]: Setup & Load Shelter Data

**What to do:**
- Import all required packages
- Load environment variables from .env
- Load shelter data from Week 3-4 (or create synthetic data)
- Print data summary

In [ ]:
# Cell [1]: YOUR CODE HERE
# 1. Import packages: geopandas, pandas, numpy, folium, requests, json, os
# 2. Load .env with python-dotenv
# 3. Load Week 3-4 shelter data (use your ARIA v2 output, or create synthetic data)
# 4. Print shelter count, CRS, and columns

# 1. Import packages
import geopandas as gpd
import pandas as pd
import numpy as np
import folium
import requests
import json
import os
from dotenv import load_dotenv

load_dotenv(override=True)
print("✅ .env loaded")

# 讀 CSV
shelter_file = "data/scenarios/shelter_Hualien_risk.csv"   # 換成你的檔名
df_shelters = pd.read_csv(shelter_file)

print("📋 原始欄位：", list(df_shelters.columns))
print("📦 原始型別：", type(df_shelters))

# 假設你的欄位叫 longitude / latitude
gdf_shelters = gpd.GeoDataFrame(
    df_shelters,
    geometry=gpd.points_from_xy(df_shelters["經度"], df_shelters["緯度"]),
    crs="EPSG:4326"
)

print(f"✅ Shelter count: {len(gdf_shelters)}")
print(f"🗺️ CRS: {gdf_shelters.crs}")
print(f"📋 Columns: {list(gdf_shelters.columns)}")

gdf_shelters.head()

## Cell [2]: Fetch CWA Rainfall API

**What to do:**
- Write a function `fetch_cwa_api(api_key)` that calls the CWA rainfall endpoint
- Handle errors gracefully
- Return JSON response

In [ ]:
# Cell [2]: YOUR CODE HERE
# Write a function fetch_cwa_api(api_key) that:
# 1. Calls https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001
# 2. Returns JSON response
# 3. Handles errors with try/except

import requests
from typing import Optional, Dict, Any

def fetch_cwa_api(api_key: str) -> Optional[Dict[str, Any]]:
    """
    呼叫 CWA 雨量 API (O-A0002-001) 並取得即時資料
    
    Args:
        api_key (str): CWA 授權碼 (Authorization Key)
        
    Returns:
        Optional[Dict[str, Any]]: 
            - 成功時回傳 API 回應的 JSON 資料 (dict)
            - 失敗時回傳 None
    """
    
    # API 端點
    base_url = "https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001"
    
    # 請求參數
    params = {
        'Authorization': api_key,
        'format': 'JSON'
    }
    
    try:
        # 發送 GET 請求
        response = requests.get(base_url, params=params, timeout=10)
        
        # 檢查 HTTP 狀態碼
        response.raise_for_status()
        
        # 解析 JSON 回應
        data = response.json()
        
        # 檢查 API 回應狀態
        if data.get("success") == "true":
            stations = data.get("records", {}).get("Station", [])
            print(f"✅ 成功取得 {len(stations)} 個雨量測站資料")
            return data
        else:
            print(f"❌ API 回應失敗")
            return None
            
    except requests.exceptions.RequestException as e:
        # 處理網路相關錯誤
        if isinstance(e, requests.exceptions.ConnectionError):
            print("❌ 網路連線失敗")
        elif isinstance(e, requests.exceptions.Timeout):
            print("❌ 請求超時")
        elif isinstance(e, requests.exceptions.HTTPError):
            if e.response.status_code == 401:
                print("❌ 認證失敗，請檢查 API 金鑰")
            elif e.response.status_code == 403:
                print("❌ 存取被拒絕")
            elif e.response.status_code == 404:
                print("❌ API 端點不存在")
            else:
                print(f"❌ HTTP 錯誤 {e.response.status_code}")
        else:
            print(f"❌ 請求錯誤")
        
        return None
        
    except Exception as e:
        print(f"❌ 未預期錯誤: {str(e)}")
        return None

## Cell [3]: Parse Rainfall JSON → GeoDataFrame

**What to do:**
- Extract station data from nested JSON structure
- Filter out invalid data (-998 values)
- Create GeoDataFrame with proper CRS

In [ ]:
# Cell [3]: YOUR CODE HERE
# Write a function parse_rainfall_json(data) that:
# 1. Detects JSON format (CWA API vs CoLife-converted JSON vs XML download)
# 2. Extracts station list from the correct root path
# 3. Gets: StationName, StationId, lat, lon, rain_1hr, rain_3hr, rain_24hr
# 4. Filters out -998 values (NoData sentinel)
# 5. Returns a GeoDataFrame with CRS EPSG:4326

# ⚠️ KEY DIFFERENCES between data sources:
#   - CWA API (format=JSON): records.Station[], Coordinates has 2 sets
#     → [0]=TWD67, [1]=WGS84. You MUST pick WGS84 or stations shift ~1km!
#   - CoLife (originally CSV, instructor converted to JSON):
#     records.Station[], Coordinates has 1 set (WGS84 only)
#   - CWA XML download: cwaopendata.dataset.Station[], 2 coordinate sets
#   - Precipitation: may be str ("130.5") or float (130.5) depending on source
#   - CountyName/TownName: inside GeoInfo, NOT at station top level

# AI Prompt suggestion:
# "Write a normalize_cwa_json(raw) function that detects the JSON format:
#  - If raw has 'records'.'Station', return that list.
#  - If raw has 'cwaopendata'.'dataset'.'Station', return that list.
#  Then write parse_rainfall_json(data) that:
#  - Calls normalize_cwa_json to get the station list
#  - For coordinates: if len(Coordinates) >= 2, find the one with
#    CoordinateName == 'WGS84'; if only 1 entry, use it directly.
#  - Convert Precipitation to float (handles both str and number).
#  - Filter out -998 sentinel values (set to 0).
#  - CountyName and TownName are inside GeoInfo, not at station level.
#  - Return GeoDataFrame with CRS EPSG:4326."

# YOUR CODE HERE
import geopandas as gpd
from shapely.geometry import Point

def normalize_cwa_json(raw):
    if raw is None:
        print("❌ raw is None")
        return []

    if 'records' in raw and 'Station' in raw['records']:
        return raw['records']['Station']

    if 'cwaopendata' in raw:
        dataset = raw['cwaopendata'].get('dataset', {})
        if 'Station' in dataset:
            return dataset['Station']

    print("❌ 無法識別 JSON 格式")
    return []


def parse_rainfall_json(data):
    stations = normalize_cwa_json(data)
    print(f"stations len = {len(stations)}")

    if not stations:
        print("❌ 沒有測站資料")
        return gpd.GeoDataFrame({'geometry': []}, geometry='geometry', crs='EPSG:4326')

    rows = []

    def to_float(v):
        try:
            v = float(v)
            return 0.0 if v == -998 else v
        except:
            return 0.0

    def safe_extract_coordinates(coords, station_name, idx):
        """安全提取座標，支援多種格式但保持原有邏輯"""
        lat, lon = None, None
        coord_system = "Unknown"

        if isinstance(coords, list) and len(coords) >= 1:
            # 優先尋找WGS84座標（如果有標示的話）
            wgs84_coord = None
            for coord in coords:
                if coord.get('CoordinateName') == 'WGS84':
                    wgs84_coord = coord
                    break
            
            # 如果找到WGS84就用它，否則用第一個（保持原有邏輯）
            c = wgs84_coord if wgs84_coord else coords[0]
            coord_system = c.get('CoordinateName', 'Unknown')
            
            # 安全地提取經緯度
            try:
                lat_str = c.get('StationLatitude', '0')
                lon_str = c.get('StationLongitude', '0')
                lat = float(lat_str) if lat_str and lat_str != '0' else None
                lon = float(lon_str) if lon_str and lon_str != '0' else None
            except ValueError:
                if idx < 5:
                    print(f"⚠️ {station_name} 座標格式錯誤，跳過此測站")
                return None, None, "Format Error"
                
        elif isinstance(coords, dict):
            # 處理字典格式的座標
            coord_system = "Dict Format"
            try:
                lat_str = coords.get('StationLatitude', '0')
                lon_str = coords.get('StationLongitude', '0')
                lat = float(lat_str) if lat_str and lat_str != '0' else None
                lon = float(lon_str) if lon_str and lon_str != '0' else None
            except ValueError:
                if idx < 5:
                    print(f"⚠️ {station_name} 座標格式錯誤，跳過此測站")
                return None, None, "Format Error"

        # 驗證座標合理性（可選，不影響原有邏輯）
        if lat is not None and lon is not None:
            if not (21.8 <= lat <= 25.3 and 119.5 <= lon <= 122.5):
                if idx < 3:  # 只報告前3個異常座標
                    print(f"📍 {station_name} 座標可能異常: ({lat}, {lon}) [{coord_system}]")
                # 仍然使用這個座標，不跳過（保持原有行為）

        return lat, lon, coord_system

    for idx, station in enumerate(stations):
        try:
            name = station.get('StationName')
            sid = station.get('StationId')

            geo = station.get('GeoInfo', {})
            county = geo.get('CountyName')
            town = geo.get('TownName')
            coords = geo.get('Coordinates', [])

            # 使用改進的座標提取函數
            lat, lon, coord_system = safe_extract_coordinates(coords, name, idx)

            if lat is None or lon is None:
                if idx < 5:
                    print(f"⚠️ {name} 缺少有效座標")
                continue

            # 提取雨量資料（保持原有邏輯）
            rain = station.get('RainfallElement', {})
            r1 = to_float(rain.get('Past1hr', {}).get('Precipitation'))
            r3 = to_float(rain.get('Past3hr', {}).get('Precipitation'))
            r24 = to_float(rain.get('Past24hr', {}).get('Precipitation'))

            # 建立資料列（保持原有格式）
            rows.append({
                'StationName': name,
                'StationId': sid,
                'latitude': lat,
                'longitude': lon,
                'rain_1hr': r1,
                'rain_3hr': r3,
                'rain_24hr': r24,
                'CountyName': county,
                'TownName': town,
                'geometry': Point(lon, lat)
            })

            # 記錄座標系統資訊（僅供參考，不影響資料）
            if idx < 3:
                print(f"✅ {name} 使用 {coord_system} 座標: ({lat:.6f}, {lon:.6f})")

        except Exception as e:
            if idx < 5:
                print(f"⚠️ 第 {idx} 筆處理失敗: {e}")
            continue

    print(f"成功轉成 rows 的筆數 = {len(rows)}")

    if not rows:
        print("❌ 沒有任何測站成功轉成 geometry")
        return gpd.GeoDataFrame({'geometry': []}, geometry='geometry', crs='EPSG:4326')

    gdf = gpd.GeoDataFrame(rows, geometry='geometry', crs='EPSG:4326')
    print(f"✅ 載入 {len(gdf)} 個測站")
    return gdf

## Cell [4]: Mode Switcher (LIVE vs SIMULATION)

**What to do:**
- Read `APP_MODE` from .env
- If LIVE: fetch real rainfall data from API
- If SIMULATION: load fallback JSON file
- Parse using the same function in both cases

In [ ]:
import os
print(os.getenv("APP_MODE"))

In [ ]:
# Cell [4]: YOUR CODE HERE
# Mode Switcher:
# 1. Read APP_MODE from .env (default: 'SIMULATION')
# 2. If LIVE: call fetch_cwa_api() → parse_rainfall_json()
# 3. If SIMULATION: load fungwong_202511.json → parse_rainfall_json()
# 4. KEY INSIGHT: same parse function for both!

# AI Prompt suggestion:
# "Read APP_MODE from os.getenv (default 'SIMULATION').
#  If LIVE, call fetch_cwa_api and parse the result.
#  If SIMULATION, load 'data/scenarios/fungwong_202511.json' and parse it.
#  Both paths use the same parse_rainfall_json function.
#  Print which mode is active and how many stations loaded."

import os
import json
import geopandas as gpd

# 1. 讀取模式，預設 SIMULATION
app_mode = os.getenv("APP_MODE", "SIMULATION").upper()
print(f"🔧 Active mode: {app_mode}")

gdf_rainfall = gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

# 2. 根據模式決定資料來源
if app_mode == "LIVE":
    print("🌐 Loading LIVE data from CWA API...")
    
    api_key = os.getenv("CWA_API_KEY")
    if not api_key:
        print("❌ CWA_API_KEY not found in environment")
    else:
        try:
            raw_data = fetch_cwa_api(api_key)
            gdf_rainfall = parse_rainfall_json(raw_data)
        except Exception as e:
            print(f"❌ Failed to fetch LIVE data: {e}")

elif app_mode == "SIMULATION":
    print("📂 Loading SIMULATION data from local JSON...")
    
    simulation_file = os.getenv(
        "SIMULATION_DATA",
        "data/scenarios/fungwong_202511.json"
    )
    
    print(f"📍 File path: {simulation_file}")
    print(f"📍 Current working directory: {os.getcwd()}")
    
    try:
        with open(simulation_file, "r", encoding="utf-8") as f:
            raw_data = json.load(f)
        gdf_rainfall = parse_rainfall_json(raw_data)
    except FileNotFoundError:
        print(f"❌ Simulation file not found: {simulation_file}")
    except json.JSONDecodeError as e:
        print(f"❌ JSON decode error: {e}")
    except Exception as e:
        print(f"❌ Failed to load simulation data: {e}")

else:
    print(f"❌ Unknown APP_MODE: {app_mode}")

# 3. 顯示結果
if not gdf_rainfall.empty:
    print(f"✅ Loaded {len(gdf_rainfall)} stations")
else:
    print("❌ No rainfall stations loaded")

## Cell [5]: Create Base Folium Map

**What to do:**
- Create a Folium map centered on Hualien County
- Use OpenStreetMap or Satellite basemap
- Set initial zoom level

In [ ]:
# Cell [5]: YOUR CODE HERE
# Create a base Folium map:
# 1. Center on Hualien (latitude ~23.98, longitude ~121.55)
# 2. Use tiles='OpenStreetMap' or tiles='Satellite'
# 3. Set zoom_start=10
# 4. Assign to variable `m`

# AI Prompt suggestion:
# "Create a Folium map centered at [23.98, 121.55] (Hualien County)
#  with zoom_start=10 and tiles='OpenStreetMap'.
#  Store it in variable m."

# YOUR CODE HERE
import folium

m = folium.Map(
    location=[23.98, 121.55],
    zoom_start=10,
    tiles="OpenStreetMap"
)

## Cell [6]: Add Rainfall CircleMarkers with Conditional Styling

**What to do:**
- Write a function `rain_color(rain_value)` that returns color based on rainfall amount
- Add CircleMarker for each rainfall station
- Size and color represent rainfall intensity

In [ ]:
# Cell [6]: YOUR CODE HERE
# 1. Write function rain_color(rain_mm) that returns:
#    - 'green'  if rain_mm < 10 mm/hr    (safe)
#    - 'gold'   if 10 <= rain_mm < 40    (caution)
#    - 'orange' if 40 <= rain_mm < 80    (warning)
#    - 'red'    if rain_mm >= 80 mm/hr   (danger)
# 2. Loop through gdf_rainfall and add CircleMarker for each station
# 3. Radius proportional to rain_1hr: radius = max(5, rain_mm / 5)

# AI Prompt suggestion:
# "Write rain_color(rain_mm) that returns 'green' (<10), 'gold' (10-40),
#  'orange' (40-80), 'red' (>=80) based on rainfall intensity.
#  Write rain_radius(rain_mm) that returns max(5, rain_mm / 5).
#  Then add CircleMarker for each station in gdf_rainfall to folium map m.
#  Use the rain_color function to set fill color and rain_radius for size.
#  Add tooltip with station name and rainfall amount."

# YOUR CODE HERE
def rain_color(rain_mm):
    if rain_mm < 10:
        return "green"
    elif rain_mm < 40:
        return "gold"
    elif rain_mm < 80:
        return "orange"
    else:
        return "red"


def rain_radius(rain_mm):
    return max(5, rain_mm / 5)


for _, row in gdf_rainfall.iterrows():
    rain_mm = row["rain_24hr"]   # 你也可以改成 rain_1hr 或 rain_3hr

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=rain_radius(rain_mm),
        color=rain_color(rain_mm),
        fill=True,
        fill_color=rain_color(rain_mm),
        fill_opacity=0.7,
        weight=1,
        tooltip=f"{row['StationName']}: {rain_mm} mm"
    ).add_to(m)

## Cell [7]: Add HeatMap Layer

**What to do:**
- Import HeatMap from folium.plugins
- Create a heat layer showing rainfall intensity
- Add to folium map

In [ ]:
# Cell [7]: YOUR CODE HERE
# 1. Import HeatMap from folium.plugins
# 2. Create list of [lat, lon, rain_1hr] for each station
# 3. Add HeatMap(data, name='Rainfall Heatmap', show=False) to map m

# AI Prompt suggestion:
# "Import HeatMap from folium.plugins.
#  Create a heat_data list of [lat, lon, rain_1hr] from gdf_rainfall.
#  Add HeatMap layer to map m with name='Rainfall Heatmap' and show=False."

# YOUR CODE HERE
from folium.plugins import HeatMap

heat_data = gdf_rainfall[["latitude", "longitude", "rain_1hr"]].values.tolist()

HeatMap(
    heat_data,
    name="Rainfall Heatmap",
    show=False
).add_to(m)

## Cell [8]: Add LayerControl

**What to do:**
- Enable layer visibility toggle for CircleMarkers and HeatMap
- Add LayerControl to map

In [ ]:
# Cell [8]: YOUR CODE HERE
# 1. Import LayerControl from folium
# 2. Add LayerControl(collapsed=False) to map m
# 3. This lets users toggle layers on/off

# AI Prompt suggestion:
# "Import LayerControl from folium.
#  Add LayerControl(collapsed=False) to map m.
#  Display the map."

# YOUR CODE HERE
from folium import LayerControl
folium.LayerControl(collapsed=False).add_to(m)

## Cell [9]: Add Shelter Risk Popups

**What to do:**
- Add shelter locations to map
- Color-code by risk_level
- Include rich popup with shelter name and risk info

In [ ]:
# Cell [9]: YOUR CODE HERE
# 1. Loop through gdf_shelters and add Marker for each shelter
# 2. Color by risk_level: 'low'→blue, 'medium'→orange, 'high'→red
# 3. Create rich popup with HTML:
#    - Shelter name
#    - Risk level
#    - Terrain risk
#    - Mean elevation
#    - Max slope

# AI Prompt suggestion:
# "Loop through gdf_shelters. For each shelter, create an HTML popup:
#  <b>{name}</b><br>Risk: {risk_level}<br>Elevation: {mean_elevation}m<br>Max Slope: {max_slope}°
#  Add Marker with icon color based on risk_level.
#  Use folium.Icon(color='blue'/'orange'/'red', icon='info-sign')."

# YOUR CODE HERE
for _, row in gdf_shelters.iterrows():
    name = row["避難收容處所名稱"]
    risk_level = row["risk_level"]
    mean_elevation = row["mean_elev"]
    max_slope = row["max_slope"]

    # popup 內容
    popup_html = f"""
    <b>{name}</b><br>
    Risk: {risk_level}<br>
    Elevation: {mean_elevation}m<br>
    Max Slope: {max_slope}°
    """

    # 根據 risk_level 設定 icon 顏色
    if risk_level == "low":
        icon_color = "blue"
    elif risk_level == "medium":
        icon_color = "blue"    
    elif risk_level == "high":
        icon_color = "orange"
    elif risk_level == "extremely_high":
        icon_color = "red"

    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=folium.Popup(popup_html, max_width=250),
        icon=folium.Icon(color=icon_color, icon="info-sign")
    ).add_to(m)
m.save("map.html")

---

# Lab 1: CWA API → Folium Map (25 minutes)

**Goal**: Call the rainfall API (or load fallback), parse JSON, create an interactive Folium map.

> **Fallback**: If CWA API doesn't work, load `data/scenarios/fungwong_202511.json` instead. The structure is similar (both use `records.Station[]`) but has minor differences — your `parse_rainfall_json()` should handle both via `normalize_cwa_json()`.

**Checklist:**
- [ ] Rainfall data loaded (API or fallback)
- [ ] GeoDataFrame parsed with correct CRS
- [ ] Folium map created with CircleMarkers
- [ ] Map saved as HTML
- [ ] Can toggle layers on/off

### Lab 1 Step 1: Load Data (API or Fallback)

In [ ]:
# Lab 1 Step 1: YOUR CODE HERE
# 1. Read API_KEY from .env
# 2. Try: fetch_cwa_api(api_key)
# 3. If error or None: load 'data/scenarios/fungwong_202511.json'
# 4. Parse JSON → gdf_rainfall
# 5. Print shape, columns, CRS

# AI Prompt suggestion:
# "Try to fetch CWA API data. If it fails, load fungwong_202511.json from
#  data/scenarios/ directory. Parse either result with parse_rainfall_json.
#  Print the GeoDataFrame info."

# YOUR CODE HERE
import os
import json

try:
    api_key = os.getenv("CWA_API_KEY")
    raw_data = fetch_cwa_api(api_key)
    print("✅ Fetched live CWA API data")

except Exception as e:
    print(f"⚠️ API failed: {e}")
    with open("data/scenarios/fungwong_202511.json", "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    print("✅ Loaded simulation JSON")

gdf_rainfall = parse_rainfall_json(raw_data)

print("Shape:", gdf_rainfall.shape)
print("Columns:", list(gdf_rainfall.columns))
print("CRS:", gdf_rainfall.crs)

### Lab 1 Step 2: Parse JSON → GeoDataFrame

In [ ]:
# Lab 1 Step 2: YOUR CODE HERE
# 1. Check gdf_rainfall has columns: rain_1hr, rain_3hr, rain_24hr
# 2. Check CRS is EPSG:4326
# 3. Display first 5 rows
# 4. Print statistics: min/max/mean rainfall

# AI Prompt suggestion:
# "Display gdf_rainfall.head(5) and gdf_rainfall.describe().
#  Check the CRS with gdf_rainfall.crs.
#  What are the highest rainfall stations?"

# YOUR CODE HERE
# 1. 顯示前 5 筆
display(gdf_rainfall.head(5))

# 2. 顯示數值統計摘要
display(gdf_rainfall.describe())

# 3. 檢查 CRS
print("CRS:", gdf_rainfall.crs)

# 4. 找出雨量最高的測站（以 24 小時雨量為例）
top_rain = gdf_rainfall.sort_values(by="rain_24hr", ascending=False)
display(top_rain[["StationName", "StationId", "CountyName", "TownName", "rain_1hr", "rain_3hr", "rain_24hr"]].head(10))

### Lab 1 Step 3: Build Folium Map + CircleMarkers

In [ ]:
# Lab 1 Step 3: YOUR CODE HERE
# 1. Create Folium map (reuse Cell [5] & [6])
# 2. Add CircleMarkers for rainfall stations
# 3. Add HeatMap layer
# 4. Add LayerControl
# 5. Display map

# AI Prompt suggestion:
# "Create a Folium map of Hualien with rainfall CircleMarkers and HeatMap.
#  Add LayerControl so users can toggle layers.
#  Display the map in the notebook."

# YOUR CODE HERE
import folium
from folium.plugins import HeatMap

# 1. 建立花蓮地圖
m = folium.Map(
    location=[23.98, 121.55],
    zoom_start=10,
    tiles="OpenStreetMap"
)

# 2. 定義顏色與半徑函式
def rain_color(rain_mm):
    if rain_mm < 10:
        return "green"
    elif rain_mm < 40:
        return "gold"
    elif rain_mm < 80:
        return "orange"
    else:
        return "red"

def rain_radius(rain_mm):
    return max(5, rain_mm / 5)

# 3. 建立 CircleMarker 圖層
circle_layer = folium.FeatureGroup(name="Rainfall Stations")

for _, row in gdf_rainfall.iterrows():
    rain_mm = row["rain_1hr"]   # 也可以改成 rain_3hr 或 rain_24hr

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=rain_radius(rain_mm),
        color=rain_color(rain_mm),
        fill=True,
        fill_color=rain_color(rain_mm),
        fill_opacity=0.6,
        weight=1,
        tooltip=f"{row['StationName']}: {rain_mm} mm"
    ).add_to(circle_layer)

circle_layer.add_to(m)

# 4. 建立 HeatMap 圖層
heat_data = gdf_rainfall[["latitude", "longitude", "rain_1hr"]].values.tolist()

heat_layer = folium.FeatureGroup(name="Rainfall Heatmap", show=False)
HeatMap(heat_data).add_to(heat_layer)
heat_layer.add_to(m)

# 5. 加上圖層切換
folium.LayerControl(collapsed=False).add_to(m)

# 6. 在 notebook 顯示地圖
m

### Lab 1 Step 4: Save Map as HTML

In [ ]:
# Lab 1 Step 4: YOUR CODE HERE
# 1. Save map to 'output/rainfall_map_week5.html'
# 2. Verify file was created
# 3. Print file size

# AI Prompt suggestion:
# "Save the map to 'output/rainfall_map_week5.html' using m.save().
#  Verify the file exists and print its size in KB."

# YOUR CODE HERE
import os

# 建立輸出資料夾（如果不存在）
os.makedirs("output", exist_ok=True)

# 存地圖
output_file = "output/rainfall_map_week5.html"
m.save(output_file)

# 檢查檔案是否存在
if os.path.exists(output_file):
    file_size_kb = os.path.getsize(output_file) / 1024
    print(f"✅ File saved: {output_file}")
    print(f"📦 File size: {file_size_kb:.2f} KB")
else:
    print(f"❌ File not found: {output_file}")

---

# 🔬 Lab 2: Typhoon Fung-wong Simulation (15 minutes)

**Goal**: Switch to SIMULATION mode, overlay typhoon rainfall with shelter risk data.

> **Context**: It's 2025-11-11 14:00. Typhoon Fung-wong is hitting eastern Taiwan.
> Suao: 130.5mm/hr. Mataian Creek is forming a landslide dam.

**Checklist:**
- [ ] Simulation data loaded
- [ ] High-rainfall stations identified
- [ ] Shelters within 5km radius found
- [ ] Risk map created and saved

### Lab 2 Step 1: Load Simulation JSON + Filter High-Rain Stations

In [ ]:
# Lab 2 Step 1: YOUR CODE HERE
# 1. Load 'data/scenarios/fungwong_202511.json'
# 2. Parse with parse_rainfall_json()
# 3. Filter: rain_1hr > 30 mm/hr (heavy rain)
# 4. Print how many high-rain stations found
# 5. Find station with max rain_1hr (Suao should be 130.5mm/hr)

# AI Prompt suggestion:
# "Load and parse the Typhoon Fung-wong simulation JSON.
#  Filter for rain_1hr > 30 mm/hr. Count how many stations exceed this.
#  Find the station with highest rainfall. Is it Suao with ~130.5mm/hr?"

# YOUR CODE HERE
import json

# 1. 載入模擬 JSON
simulation_file = "data/scenarios/fungwong_202511.json"

with open(simulation_file, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# 2. 解析成 GeoDataFrame
gdf_fungwong = parse_rainfall_json(raw_data)

# 3. 篩選 rain_1hr > 30 mm/hr
heavy_rain = gdf_fungwong[gdf_fungwong["rain_1hr"] > 30]

# 4. 計算超過站數
print(f"rain_1hr > 30 mm/hr 的測站數量: {len(heavy_rain)}")

# 5. 找最高 1 小時雨量站
top_station = gdf_fungwong.sort_values(by="rain_1hr", ascending=False).iloc[0]

print("\n最高 1 小時雨量測站:")
print(f"StationName: {top_station['StationName']}")
print(f"StationId: {top_station['StationId']}")
print(f"CountyName: {top_station['CountyName']}")
print(f"TownName: {top_station['TownName']}")
print(f"rain_1hr: {top_station['rain_1hr']} mm/hr")

# 6. 檢查是否為蘇澳、約 130.5 mm/hr
is_suao = "蘇澳" in str(top_station["StationName"])
is_130_5 = abs(top_station["rain_1hr"] - 130.5) < 1e-6

print("\n檢查結果:")
print(f"是否為蘇澳: {is_suao}")
print(f"是否約為 130.5 mm/hr: {is_130_5}")

## 人造避難所

In [ ]:
# Lab 2 Step 2: YOUR CODE HERE
# 1. CRITICAL: Reproject gdf_rainfall to EPSG:3826 (same as shelters)
# 2. Filter high-rain stations (rain_1hr > 40mm)
# 3. Create 5km buffer around high-rain stations
# 4. Use gpd.sjoin(gdf_shelters, buffered_rain, how='left', predicate='within')
#    to find shelters inside the 5km impact zones
# 5. Flag shelters at risk and assign dynamic risk level

# AI Prompt suggestion:
# "Reproject gdf_rainfall to EPSG:3826 (meters).
#  Filter stations where rain_1hr > 40.
#  Create a 5000m buffer around each high-rain station geometry.
#  Make a new GeoDataFrame with the buffer as geometry.
#  Use gpd.sjoin with predicate='within' to find shelters inside buffers.
#  Apply risk logic: CRITICAL if rain > 80mm, URGENT if rain > 40mm
#  and terrain_risk == 'HIGH', WARNING otherwise."

# YOUR CODE HERE
import geopandas as gpd
import pandas as pd

# 1. 重投影到 EPSG:3826（公尺）- 使用 gdf_fungwong 而不是 gdf_rainfall
gdf_rainfall_3826 = gdf_fungwong.to_crs(epsg=3826)  # 修正這裡
gdf_shelters_3826 = gdf_shelters.to_crs(epsg=3826)

print("Rainfall CRS:", gdf_rainfall_3826.crs)
print("Shelter CRS:", gdf_shelters_3826.crs)

# 如果有舊的 risk_level，先刪掉
if "risk_level" in gdf_shelters_3826.columns:
    gdf_shelters_3826 = gdf_shelters_3826.drop(columns=["risk_level"])

# 2. 篩選 1 小時雨量 > 40 mm 的測站
high_rain_stations = gdf_rainfall_3826[gdf_rainfall_3826["rain_1hr"] > 40].copy()
print(f"高雨量測站數量 (>40 mm/hr): {len(high_rain_stations)}")

# 3. 建立 5000m buffer
high_rain_stations["buffer_geom"] = high_rain_stations.geometry.buffer(5000)

# 4. 建立新的 buffer GeoDataFrame
#    把 rain_1hr 改名成 station_rain_1hr，避免後面欄位混淆
buffer_gdf = gpd.GeoDataFrame(
    high_rain_stations[["StationName", "StationId", "rain_1hr"]].copy(),
    geometry=high_rain_stations["buffer_geom"],
    crs="EPSG:3826"
).rename(columns={"rain_1hr": "station_rain_1hr"})

print("Buffer GDF shape:", buffer_gdf.shape)

# 5. 空間連接：保留所有 shelters，因此用 how='left'
shelters_joined = gpd.sjoin(
    gdf_shelters_3826,
    buffer_gdf,
    how="left",
    predicate="within"
).copy()

print(f"sjoin 後資料筆數: {len(shelters_joined)}")

# 6. 套用新的動態風險邏輯
def assign_risk_level(row):
    rain = row["station_rain_1hr"]

    if pd.isna(rain):
        return "SAFE"
    elif rain > 80:
        return "CRITICAL"
    elif rain > 40:
        return "URGENT"
    else:
        return "WARNING"

shelters_joined["risk_level"] = shelters_joined.apply(assign_risk_level, axis=1)
shelters_joined["at_risk"] = shelters_joined["risk_level"] != "SAFE"

# 7. 如果同一 shelter 落在多個 buffer 內，保留最高風險
risk_order = {"SAFE": 0, "WARNING": 1, "URGENT": 2, "CRITICAL": 3}
shelters_joined["risk_score"] = shelters_joined["risk_level"].map(risk_order)

# 你的唯一識別欄位先用「避難收容處所地址」
# 如果之後發現地址不唯一，再改成別的欄位
shelters_final = (
    shelters_joined
    .sort_values("risk_score", ascending=False)
    .drop_duplicates(subset=["避難收容處所地址"])
    .drop(columns=["risk_score"])
    .copy()
)

print(f"去除重複後的 shelters 數量: {len(shelters_final)}")

# 8. 查看結果
# 只印確定存在的欄位
cols_to_show = [
    "避難收容處所地址",
    "StationName",
    "station_rain_1hr",
    "risk_level",
    "at_risk"
]

existing_cols = [c for c in cols_to_show if c in shelters_final.columns]
print(shelters_final[existing_cols].head(10))

### Lab 2 Step 2: Spatial Join Rainfall with Shelters

In [ ]:
# Lab 2 Step 2: YOUR CODE HERE
# 1. CRITICAL: Reproject gdf_rainfall to EPSG:3826 (same as shelters)
# 2. Filter high-rain stations (rain_1hr > 40mm)
# 3. Create 5km buffer around high-rain stations
# 4. Use gpd.sjoin(gdf_shelters, buffered_rain, how='left', predicate='within')
#    to find shelters inside the 5km impact zones
# 5. Flag shelters at risk and assign dynamic risk level

# AI Prompt suggestion:
# "Reproject gdf_rainfall to EPSG:3826 (meters).
#  Filter stations where rain_1hr > 40.
#  Create a 5000m buffer around each high-rain station geometry.
#  Make a new GeoDataFrame with the buffer as geometry.
#  Use gpd.sjoin with predicate='within' to find shelters inside buffers.
#  Apply risk logic: CRITICAL if rain > 80mm, URGENT if rain > 40mm
#  and terrain_risk == 'HIGH', WARNING otherwise."

# YOUR CODE HERE
import os
import folium
from folium.plugins import HeatMap

# 1. 建立地圖（改成宜蘭附近，比較容易看到 synthetic shelters）
m_risk = folium.Map(
    location=[24.67, 121.78],   # 宜蘭/羅東/蘇澳附近
    zoom_start=11,
    tiles="OpenStreetMap"
)

# 2. 雨量顏色與大小函式
def rain_color(rain_mm):
    if rain_mm < 10:
        return "green"
    elif rain_mm < 40:
        return "gold"
    elif rain_mm < 80:
        return "orange"
    else:
        return "red"

def rain_radius(rain_mm):
    return max(5, rain_mm / 5)

# 3. 加入颱風雨量 CircleMarker
rain_layer = folium.FeatureGroup(name="Typhoon Fung-wong Rainfall Stations")

# 建議只畫較明顯的站，避免全台太密
rain_to_plot = gdf_fungwong[gdf_fungwong["rain_1hr"] > 10].copy()

for _, row in rain_to_plot.iterrows():
    rain_mm = row["rain_1hr"]

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=rain_radius(rain_mm),
        color=rain_color(rain_mm),
        fill=True,
        fill_color=rain_color(rain_mm),
        fill_opacity=0.5,
        weight=1,
        tooltip=f"{row['StationName']} | 1hr rain: {rain_mm} mm"
    ).add_to(rain_layer)

rain_layer.add_to(m_risk)

# 4. 加入 HeatMap
heat_data = gdf_fungwong[["latitude", "longitude", "rain_1hr"]].values.tolist()

heat_layer = folium.FeatureGroup(name="Rainfall Heatmap", show=False)
HeatMap(heat_data).add_to(heat_layer)
heat_layer.add_to(m_risk)

# 5. 加入 shelter 風險 Marker
shelter_layer = folium.FeatureGroup(name="Shelter Risk")

if len(shelters_final) == 0:
    print("⚠️ shelters_final 是空的，沒有 shelter 可繪製")
else:
    shelters_final_4326 = shelters_final.to_crs(epsg=4326)

    for _, row in shelters_final_4326.iterrows():
        at_risk = row["at_risk"]
        risk_level = row["risk_level"]

        # 高風險紅色，其餘藍色
        icon_color = "red" if at_risk else "blue"

        rain_value = row.get("station_rain_1hr", None)
        if rain_value is None or str(rain_value) == "nan":
            rain_text = "N/A"
        else:
            rain_text = f"{rain_value} mm/hr"

        popup_html = f"""
        <b>避難所</b><br>
        地址: {row.get('避難收容處所地址', 'N/A')}<br>
        Risk Level: {risk_level}<br>
        At Risk: {at_risk}<br>
        Rainfall: {rain_text}
        """

        folium.Marker(
            location=[row.geometry.y, row.geometry.x],
            popup=folium.Popup(popup_html, max_width=300),
            icon=folium.Icon(color=icon_color, icon="info-sign")
        ).add_to(shelter_layer)

shelter_layer.add_to(m_risk)

# 6. 加上圖層控制
folium.LayerControl(collapsed=False).add_to(m_risk)

# 7. 輸出 HTML
os.makedirs("output", exist_ok=True)
output_file = "output/typhoon_fungwong_risk_map.html"
m_risk.save(output_file)

# 8. 統計高風險避難所數量
high_risk_count = int(shelters_final["at_risk"].sum()) if len(shelters_final) > 0 else 0

print(f"{high_risk_count} shelters are at HIGH RISK during Typhoon Fung-wong.")
print(f"Map saved to: {output_file}")

# 9. 顯示地圖
m_risk

### Lab 2 Step 3: Final Map + Save HTML

In [ ]:
# Lab 2 Step 3: YOUR CODE HERE
# 1. Create new Folium map (same center/zoom as Lab 1)
# 2. Add rainfall CircleMarkers (heavy rain = bigger, redder)
# 3. Add shelter Markers:
#    - Blue if low/medium risk
#    - Red if high_risk flag = True
# 4. Add HeatMap + LayerControl
# 5. Save to 'output/typhoon_fungwong_risk_map.html'
# 6. Display statistics: how many shelters are at risk?

# AI Prompt suggestion:
# "Create a new Folium map showing both rainfall and shelter risk.
#  Highlight shelters in red if they're within 5km of heavy rainfall.
#  Save to output/typhoon_fungwong_risk_map.html.
#  Print: 'X shelters are at HIGH RISK during Typhoon Fung-wong.'"

# YOUR CODE HERE
import os
import folium
from folium.plugins import HeatMap

# 1. 建立地圖（以花蓮附近為中心）
m_risk = folium.Map(
    location=[23.98, 121.55],
    zoom_start=9,
    tiles="OpenStreetMap"
)

# 2. 雨量顏色與大小函式
def rain_color(rain_mm):
    if rain_mm < 10:
        return "green"
    elif rain_mm < 40:
        return "gold"
    elif rain_mm < 80:
        return "orange"
    else:
        return "red"

def rain_radius(rain_mm):
    return max(5, rain_mm / 5)

# 3. 加入颱風雨量 CircleMarker - 使用 gdf_fungwong 而不是 gdf_rainfall
rain_layer = folium.FeatureGroup(name="Typhoon Fung-wong Rainfall")

for _, row in gdf_fungwong.iterrows():
    rain_mm = row["rain_1hr"]

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=rain_radius(rain_mm),
        color=rain_color(rain_mm),
        fill=True,
        fill_color=rain_color(rain_mm),
        fill_opacity=0.5,
        weight=1,
        tooltip=f"{row['StationName']} | 1hr rain: {rain_mm} mm"
    ).add_to(rain_layer)

rain_layer.add_to(m_risk)

# 4. 加入 HeatMap - 使用颱風數據
heat_data = gdf_fungwong[["latitude", "longitude", "rain_1hr"]].values.tolist()

heat_layer = folium.FeatureGroup(name="Rainfall Heatmap", show=False)
HeatMap(heat_data).add_to(heat_layer)
heat_layer.add_to(m_risk)

# 5. 加入避難所風險 Marker
shelter_layer = folium.FeatureGroup(name="Shelter Risk")

# 檢查 shelters_final 是否為空
if len(shelters_final) == 0:
    print("⚠️ shelters_final 是空的，使用原始 gdf_shelters")
    # 如果 shelters_final 是空的，使用原始的 gdf_shelters
    shelters_to_plot = gdf_shelters.to_crs(epsg=4326)
    for _, row in shelters_to_plot.iterrows():
        popup_html = f"""
        <b>避難所</b><br>
        地址: {row.get('避難收容處所地址', 'N/A')}<br>
        Risk: 原始數據
        """
        folium.Marker(
            location=[row.geometry.y, row.geometry.x],
            popup=folium.Popup(popup_html, max_width=300),
            icon=folium.Icon(color="blue", icon="info-sign")
        ).add_to(shelter_layer)
else:
    # 使用 shelters_final
    shelters_final_4326 = shelters_final.to_crs(epsg=4326)
    for _, row in shelters_final_4326.iterrows():
        risk_level = row["risk_level"]
        
        # 高風險用紅色，其餘藍色
        icon_color = "red" if risk_level in ["URGENT", "CRITICAL"] else "blue"
        
        popup_html = f"""
        <b>避難所</b><br>
        地址: {row.get('避難收容處所地址', 'N/A')}<br>
        Risk Level: {risk_level}<br>
        At Risk: {row.get('at_risk', 'N/A')}<br>
        Rainfall: {row.get('station_rain_1hr', 'N/A')} mm/hr
        """
        
        folium.Marker(
            location=[row.geometry.y, row.geometry.x],
            popup=folium.Popup(popup_html, max_width=300),
            icon=folium.Icon(color=icon_color, icon="info-sign")
        ).add_to(shelter_layer)

shelter_layer.add_to(m_risk)

# 6. 加上圖層控制
folium.LayerControl(collapsed=False).add_to(m_risk)

# 7. 輸出 HTML
os.makedirs("output", exist_ok=True)
output_file = "output/typhoon_fungwong_risk_map.html"
m_risk.save(output_file)

# 8. 統計高風險避難所數量
if len(shelters_final) > 0:
    high_risk_count = shelters_final["risk_level"].isin(["URGENT", "CRITICAL"]).sum()
    print(f"{high_risk_count} shelters are at HIGH RISK during Typhoon Fung-wong.")
else:
    print("⚠️ No shelters in risk analysis - showing original shelter locations")

print(f"Map saved to: {output_file}")

# 9. 顯示地圖
m_risk

---

# 💭 My Reflection (fill in your answers)

### 1. How many lines of code did you change between LIVE and SIMULATION mode?

*只需要改1-2行程式碼。主要差別是將空間分析和地圖函數中的資料來源變數從 `gdf_rainfall` 改為 `gdf_fungwong`。這個設計的美妙之處在於同樣的 `parse_rainfall_json()` 函數可以處理兩種不同的資料來源。*

---

### 2. What happens if you forget to convert CRS before `sjoin`?

*如果在 `sjoin` 前忘記轉換 CRS，空間連接會失敗或產生錯誤結果，因為資料集使用不同的座標參考系統。例如，如果避難所是 EPSG:3826（公尺）而雨量資料是 EPSG:4326（度），距離計算和空間關係會是錯誤的。這可能導致避難所被錯誤分類為安全或有風險。*

---

### 3. Why does CWA use -998 instead of NaN or null?

*CWA 使用 -998 作為哨兵值來表示 "NoData" 或缺失測量值。這在氣象和科學資料系統中是常見做法，因為：
- 這是一個獨特的值，不會與真實測量值混淆
- 它在不同資料格式和系統中都能運作，即使某些系統不支援 NaN/null
- 它允許輕易識別和過濾無效讀數
- 這是確保與舊系統相容性的遺留標準*

---

### 4. During Typhoon Fung-wong, which shelter would you evacuate first and why?

*我會優先撤離蘇澳區的避難所，特別是：
- 蘇澳鎮中山路1號
- 蘇澳鎮港區路15號  
- 蘇澳鎮聖湖路8號

原因：這些避難所被分類為 "CRITICAL" 風險，因為它們在蘇澳測站5公里範圍內，該測站記錄到130.5mm/小时的降雨量——這是極端、危及生命的降雨率。極端降雨和山區潛在山崩/暴洪的結合，使這些地點最危險，需要立即撤離。*

---

### 5. What challenges did you face in this lab? How did you solve them?

*遇到的挑戰：
1. 變數混淆：最初在颱風分析中使用 `gdf_rainfall` 而不是 `gdf_fungwong`，導致 `shelters_final` 為空
2. 資料來源不一致：不同cell使用不同的雨量資料集（LIVE vs SIMULATION）
3. CRS轉換問題：需要確保兩個資料集在空間分析前使用相同的座標系統

解決方案：
1. 變數追蹤：仔細追蹤哪個變數包含颱風資料（`gdf_fungwong`）並在所有颱風相關分析中一致使用
2. 程式碼一致性：更新所有地圖和分析函數使用正確的颱風資料集
3. CRS標準化：在空間操作前明確將雨量和避難所資料轉換為EPSG:3826
4. 除錯輸出：加入print陳述式來追蹤資料流並識別變數何時被重置或錯誤使用

重要學習：在同一個notebook中使用多個資料集時，變數命名一致性和資料來源追蹤的重要性。*